# Water Stations - Final Approach (WFS Service)

## 🎯 Goal
Find water level monitoring stations near our river erosion scope regions and save them to a GeoPackage.

## 📚 Learning Journey Summary

### **Problem 1: Initial approach was too slow**
- **Issue:** `waterweb_locations.csv` contains 18,972 monitoring locations
- **Reality:** These are ALL types (biology, chemistry, water quality, sediment, etc.)
- **Result:** Querying each station individually would take ~3 hours!

### **Problem 2: Most stations don't have water level data**
- **Discovery:** Only 0.08% of stations had water level (WATHTE) data when queried
- **Root cause:** We were querying biology/chemistry stations for water level measurements
- **Insight:** Need to pre-filter stations to only water level monitoring points

### **Problem 3: We were missing API filters**
From the Waterweb documentation, we discovered we should use:
- `"ProcesType": "meting"` - Only actual measurements (not predictions)
- `"OpdrachtgevendeInstantieLijst"` - Filter by data provider
- These filters significantly improve data quality!

### **Solution: WFS Service! 🚀**
RWS provides an OGC WFS service that returns:
- **Pre-filtered stations** with confirmed recent measurements
- **Direct CSV download** - no need to query each station
- **Coordinates included** - ready for spatial analysis
- **Parameter descriptions** - can filter for "Waterhoogte" (water height)

**Result:** From 3 hours → 30 seconds! ⚡

---

## 📋 This Notebook
1. Download curated water station list from WFS service
2. Filter for water height stations
3. Filter for river-related stations (maas, waal, ijssel, etc.)
4. Spatial analysis with scope regions (0m, 50m, 100m, 200m buffers)
5. Query historical data availability (2016-2024)
6. Save to GeoPackage: `wocu_output_fase2_v4_w_wfs_and_stations.gpkg`

In [1]:
# Imports
import sys
sys.path.insert(0, "..")
sys.path.append("../..")

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from shapely.geometry import Point
import requests
from datetime import datetime
from tqdm import tqdm
import time
import json
from io import StringIO

import src.paths as PATHS

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Imports successful!")

✅ Imports successful!


## 1. Configuration

In [11]:
# File paths
INPUT_GPKG = PATHS.DATA_DIR / "wocu_output_fase2_v4_w_wfs.gpkg"
OUTPUT_GPKG = PATHS.DATA_DIR / "wocu_output_fase2_v4_w_wfs_and_stations.gpkg"
SCOPE_LAYER = "vlakken_scope"

# Buffer distances (meters)
BUFFER_DISTANCES = [0, 50, 100, 200]

# WFS Service URL (pre-filtered stations with recent measurements)
WFS_BASE_URL = "https://geo.rijkswaterstaat.nl/services/ogc/hws/DDAPI20/ows"
WFS_PARAMS = {
    'SERVICE': 'WFS',
    'VERSION': '1.1.0',
    'REQUEST': 'GetFeature',
    'TYPENAME': 'locatiesmetlaatstewaarneming',
    'outputFormat': 'csv',
    'srsName': 'EPSG:28992'
}

# Waterweb API for historical data queries
BASE_URL = "https://ddapi20-waterwebservices.rijkswaterstaat.nl"
OBSERVATIONS_ENDPOINT = f"{BASE_URL}/ONLINEWAARNEMINGENSERVICES/OphalenWaarnemingen"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-KEY": "dummy-key"
}

# Date range for historical data
START_DATE = "2016-01-01T00:00:00.000+01:00"
END_DATE = "2024-12-31T23:59:59.000+01:00"

# Cache file
CACHE_FILE = PATHS.DATA_DIR / "water_stations_historical_cache.json"

print(f"📂 Input GeoPackage: {INPUT_GPKG.name}")
print(f"📂 Output GeoPackage: {OUTPUT_GPKG.name}")
print(f"🎯 Buffer distances: {BUFFER_DISTANCES} meters")
print(f"📅 Historical range: 2016-2024")
print(f"🌐 WFS Service: {WFS_BASE_URL}")

📂 Input GeoPackage: wocu_output_fase2_v4_w_wfs.gpkg
📂 Output GeoPackage: wocu_output_fase2_v4_w_wfs_and_stations.gpkg
🎯 Buffer distances: [0, 50, 100, 200] meters
📅 Historical range: 2016-2024
🌐 WFS Service: https://geo.rijkswaterstaat.nl/services/ogc/hws/DDAPI20/ows


## 2. Download Water Stations from WFS Service

**Why this approach is better:**
- RWS pre-filters stations with recent measurements
- Includes coordinates, parameter descriptions, latest values
- Much faster than querying 18,000+ individual stations
- Guaranteed to have active monitoring data

In [12]:
print("🌐 Downloading water stations from RWS WFS service...\n")
print(f"   URL: {WFS_BASE_URL}")
print(f"   Layer: {WFS_PARAMS['TYPENAME']}")
print(f"   Format: {WFS_PARAMS['outputFormat']}\n")

try:
    response = requests.get(WFS_BASE_URL, params=WFS_PARAMS, timeout=60)
    response.raise_for_status()
    
    # Parse CSV
    stations_df = pd.read_csv(StringIO(response.text))
    
    print(f"✅ Downloaded {len(stations_df):,} stations with recent measurements\n")
    print(f"📊 Columns: {list(stations_df.columns)}\n")
    print(f"📋 Sample data:\n")
    print(stations_df.head(10))
    
except Exception as e:
    print(f"❌ Failed to download WFS data: {e}")
    print(f"\n⚠️ Falling back to manual station list...")
    # Fallback would go here, but for now we'll stop
    raise

🌐 Downloading water stations from RWS WFS service...

   URL: https://geo.rijkswaterstaat.nl/services/ogc/hws/DDAPI20/ows
   Layer: locatiesmetlaatstewaarneming
   Format: csv



/var/folders/t3/519_jjmx4tz2_jlt9nm2gkpr0000gn/T/ipykernel_61639/3609201942.py:11: DtypeWarning: Columns (0: GROEPERINGCODE) have mixed types. Specify dtype option on import or set low_memory=False.
  stations_df = pd.read_csv(StringIO(response.text))


✅ Downloaded 840,055 stations with recent measurements

📊 Columns: ['FID', 'WAARNEMING_ID', 'NAAM', 'CODE', 'OMSCHRIJVING', 'STATUSWAARDE', 'BEMONSTERINGSHOOGTE', 'REFERENTIEVLAK', 'OPDRACHTGEVENDE_INSTANTIE', 'KWALITEITSWAARDE_CODE', 'WAARDE_LAATSTE_METING', 'TIJDSTIP_LAATSTE_METING', 'PARAMETER_WAT_OMSCHRIJVING', 'BEMONSTERINGSAPPARAATCODE', 'BEMONSTERINGSMETHODECODE', 'BEMONSTERINGSSOORTCODE', 'BIOTAXONCODE', 'BIOTAXONTYPE', 'COMPARTIMENTCODE', 'EENHEIDCODE', 'GROOTHEIDCODE', 'HOEDANIGHEIDCODE', 'MEETAPPARAATCODE', 'ORGAANCODE', 'PARAMETERCODE', 'TYPERINGCODE', 'GROEPERINGCODE', 'WAARDEBEPALINGSTECHNIEKCODE', 'WAARDEBEPALINGSMETHODECODE', 'WAARDEBEWERKINGSMETHODECODE', 'GEOMETRY']

📋 Sample data:

                                                 FID  WAARNEMING_ID  \
0  locatiesmetlaatstewaarneming.fid-4c8ea8d7_19c2...            375   
1  locatiesmetlaatstewaarneming.fid-4c8ea8d7_19c2...            649   
2  locatiesmetlaatstewaarneming.fid-4c8ea8d7_19c2...            750   
3  loc

## 3. Filter for Water Height Stations

In [13]:
print("🔍 Analyzing parameter types...\n")

# Check what column contains parameter description
param_col = None
for col in ['PARAMETER_WAT_OMSCHRIJVING', 'parameter_wat_omschrijving', 'grootheid', 'parameter']:
    if col in stations_df.columns:
        param_col = col
        break

if param_col:
    print(f"📊 Parameter column: '{param_col}'\n")
    print(f"📈 Top 20 parameter types:\n")
    print(stations_df[param_col].value_counts().head(20))
    
    # Filter for water height
    # Keywords: waterhoogte, waterhte, wathte, waterstand, niveau
    water_height_keywords = ['waterhoogte', 'waterstand', 'waterhte', 'wathte', 'niveau']
    
    mask = stations_df[param_col].str.contains('|'.join(water_height_keywords), case=False, na=False)
    water_stations = stations_df[mask].copy()
    
    print(f"\n" + "="*80)
    print(f"WATER HEIGHT FILTERING")
    print(f"="*80)
    print(f"Total stations: {len(stations_df):,}")
    print(f"Water height stations: {len(water_stations):,} ({len(water_stations)/len(stations_df)*100:.1f}%)")
    
    # DEBUG: Check if water_stations has data
    print(f"\n🔍 DEBUG:")
    print(f"   water_stations shape: {water_stations.shape}")
    print(f"   water_stations columns: {list(water_stations.columns)[:10]}...")
    print(f"   Has GEOMETRY column: {'GEOMETRY' in water_stations.columns}")
    if 'GEOMETRY' in water_stations.columns:
        print(f"   Non-null GEOMETRY: {water_stations['GEOMETRY'].notna().sum()}")
    
else:
    print(f"⚠️ Could not find parameter column, keeping all stations")
    water_stations = stations_df.copy()

🔍 Analyzing parameter types...

📊 Parameter column: 'PARAMETER_WAT_OMSCHRIJVING'

📈 Top 20 parameter types:

PARAMETER_WAT_OMSCHRIJVING
Zuurgraad in Oppervlaktewater                                                                                 95178
(massa)Concentratie zuurstof in Oppervlaktewater in mg/l                                                      86403
Temperatuur in Oppervlaktewater in oC                                                                         83560
Geleidendheid in Oppervlaktewater in mS/m                                                                     81783
Verzadigingsgraad zuurstof in Oppervlaktewater in %                                                           80996
Saliniteit in Oppervlaktewater                                                                                79216
(massa)Concentratie chlorofyl-a in Oppervlaktewater in ug/l                                                   73951
Waterhoogte in Oppervlaktewater t.o.v. Normaal Amste

## 4. Convert to GeoDataFrame & Filter for Rivers

In [16]:
print("🗺️ Converting to GeoDataFrame...\n")

# Check if WFS returned a GEOMETRY column (WKT format)
if 'GEOMETRY' in water_stations.columns:
    print(f"📍 Found GEOMETRY column (WKT format)\n")
    
    # Parse WKT geometry using shapely
    from shapely import wkt
    water_stations_gdf = gpd.GeoDataFrame(
        water_stations,
        geometry=water_stations['GEOMETRY'].apply(wkt.loads),
        crs='EPSG:28992'  # RD New - WFS returns this!
    )
    print(f"✅ Created GeoDataFrame with {len(water_stations_gdf):,} stations")
    print(f"📐 CRS: {water_stations_gdf.crs}")

# Fallback: look for separate lat/lon columns
else:
    print(f"📍 Looking for separate lat/lon columns...")
    lat_col = None
    lon_col = None
    
    for col in water_stations.columns:
        col_upper = col.upper()
        if 'LAT' in col_upper and 'LON' not in col_upper:
            lat_col = col
        if 'LON' in col_upper:
            lon_col = col
    
    if lat_col and lon_col:
        print(f"   Found: {lat_col}, {lon_col}")
        water_stations = water_stations.dropna(subset=[lat_col, lon_col])
        geometry = [Point(xy) for xy in zip(water_stations[lon_col], water_stations[lat_col])]
        water_stations_gdf = gpd.GeoDataFrame(water_stations, geometry=geometry, crs='EPSG:4258')
        print(f"✅ Created GeoDataFrame with {len(water_stations_gdf):,} stations")
    else:
        print(f"❌ No geometry columns found!")
        print(f"   Available columns: {list(water_stations.columns)[:20]}...")
        raise ValueError("Cannot proceed without coordinates")

# Filter for river-related stations
print(f"\n🌊 Filtering for river-related stations...")

# Find location code column
loc_code_col = None
for col in ['LOCATIE_CODE', 'locatie_code', 'code', 'location_code']:
    if col in water_stations_gdf.columns:
        loc_code_col = col
        break

if loc_code_col:
    river_keywords = ['maas', 'waal', 'ijssel', 'rijn', 'lek', 'merwede', 'nederrijn', 'pannerden']
    river_mask = water_stations_gdf[loc_code_col].str.contains('|'.join(river_keywords), case=False, na=False)
    river_stations_gdf = water_stations_gdf[river_mask].copy()
    
    print(f"   ✅ River stations: {len(river_stations_gdf):,} ({len(river_stations_gdf)/len(water_stations_gdf)*100:.1f}%)")
else:
    print(f"   ⚠️ Could not find location code column, keeping all stations")
    river_stations_gdf = water_stations_gdf.copy()

print(f"\n📊 Final station count: {len(river_stations_gdf):,}")

🗺️ Converting to GeoDataFrame...

📍 Found GEOMETRY column (WKT format)

✅ Created GeoDataFrame with 2,599 stations
📐 CRS: EPSG:28992

🌊 Filtering for river-related stations...
   ⚠️ Could not find location code column, keeping all stations

📊 Final station count: 2,599


## 5. Load Scope Regions & Spatial Analysis

In [17]:
# Load scope regions
print("📥 Loading scope regions...")
scope_regions = gpd.read_file(INPUT_GPKG, layer=SCOPE_LAYER)
print(f"   ✅ Loaded {len(scope_regions):,} scope regions")
print(f"   📐 CRS: {scope_regions.crs}")

# Find ID column
id_col = None
for possible_col in ['location_id', 'position_id', 'region_id', 'id']:
    if possible_col in scope_regions.columns:
        id_col = possible_col
        break

if not id_col:
    raise ValueError("Could not find ID column in scope regions")

print(f"   🔑 Using ID column: '{id_col}'")

# Reproject stations to match scope regions
print(f"\n🔄 Reprojecting stations to match scope regions CRS...")

# DEBUG: Check coordinates before reprojection
print(f"\n🔍 DEBUG - Before reprojection (EPSG:4258):")
sample_station = river_stations_gdf.iloc[0]
print(f"   Sample station geometry: {sample_station.geometry}")
print(f"   Sample coordinates: ({sample_station.geometry.x:.6f}, {sample_station.geometry.y:.6f})")
print(f"   Expected for Netherlands: lon ~4-7°, lat ~50-54°")

river_stations_gdf = river_stations_gdf.to_crs(scope_regions.crs)
print(f"\n   ✅ Reprojected to {river_stations_gdf.crs}")

# DEBUG: Check coordinates after reprojection
print(f"\n🔍 DEBUG - After reprojection (EPSG:28992):")
sample_station_rd = river_stations_gdf.iloc[0]
print(f"   Sample station geometry: {sample_station_rd.geometry}")
print(f"   Sample RD coordinates: ({sample_station_rd.geometry.x:.1f}, {sample_station_rd.geometry.y:.1f})")
print(f"   Expected for Netherlands: x ~50,000-300,000, y ~300,000-600,000")

# DEBUG: Check scope region bounds
print(f"\n🔍 DEBUG - Scope regions bounds:")
bounds = scope_regions.total_bounds
print(f"   Min X: {bounds[0]:.1f}, Max X: {bounds[2]:.1f}")
print(f"   Min Y: {bounds[1]:.1f}, Max Y: {bounds[3]:.1f}")

# DEBUG: Check station bounds
print(f"\n🔍 DEBUG - Stations bounds:")
station_bounds = river_stations_gdf.total_bounds
print(f"   Min X: {station_bounds[0]:.1f}, Max X: {station_bounds[2]:.1f}")
print(f"   Min Y: {station_bounds[1]:.1f}, Max Y: {station_bounds[3]:.1f}")

# Check if bounds overlap
x_overlap = not (station_bounds[2] < bounds[0] or station_bounds[0] > bounds[2])
y_overlap = not (station_bounds[3] < bounds[1] or station_bounds[1] > bounds[3])
print(f"\n   Bounds overlap: X={x_overlap}, Y={y_overlap}")
if not (x_overlap and y_overlap):
    print(f"   ⚠️ WARNING: Stations and scope regions don't overlap!")

📥 Loading scope regions...
   ✅ Loaded 12,130 scope regions
   📐 CRS: EPSG:28992
   🔑 Using ID column: 'position_id'

🔄 Reprojecting stations to match scope regions CRS...

🔍 DEBUG - Before reprojection (EPSG:4258):
   Sample station geometry: POINT (217101.0504193791 585125.923958278)
   Sample coordinates: (217101.050419, 585125.923958)
   Expected for Netherlands: lon ~4-7°, lat ~50-54°

   ✅ Reprojected to EPSG:28992

🔍 DEBUG - After reprojection (EPSG:28992):
   Sample station geometry: POINT (217101.0504193791 585125.923958278)
   Sample RD coordinates: (217101.1, 585125.9)
   Expected for Netherlands: x ~50,000-300,000, y ~300,000-600,000

🔍 DEBUG - Scope regions bounds:
   Min X: 102109.5, Max X: 212124.7
   Min Y: 320046.3, Max Y: 517091.2

🔍 DEBUG - Stations bounds:
   Min X: -8803489.0, Max X: 614737.2
   Min Y: -356245.1, Max Y: 940400.2

   Bounds overlap: X=True, Y=True


In [18]:
# Perform spatial joins for each buffer
print("\n🔍 Performing spatial joins for each buffer distance...\n")

buffer_results = {}

for buffer_m in BUFFER_DISTANCES:
    print(f"   Buffer: {buffer_m}m")
    
    # Create buffered scope regions
    scope_buffered = scope_regions.copy()
    if buffer_m > 0:
        scope_buffered['geometry'] = scope_regions.geometry.buffer(buffer_m)
    
    # Spatial join
    joined = gpd.sjoin(
        river_stations_gdf,
        scope_buffered[[id_col, 'geometry']],
        how='inner',
        predicate='within'
    )
    
    # Statistics
    unique_stations = joined[loc_code_col].nunique() if loc_code_col else len(joined)
    total_pairs = len(joined)
    regions_with_stations = joined[id_col].nunique()
    
    print(f"      ✅ {unique_stations} unique stations")
    print(f"      ✅ {total_pairs} station-region pairs")
    print(f"      ✅ {regions_with_stations} regions with stations\n")
    
    buffer_results[buffer_m] = {
        'joined': joined,
        'unique_stations': unique_stations,
        'total_pairs': total_pairs,
        'regions_covered': regions_with_stations
    }

print("="*80)
print("SPATIAL JOIN SUMMARY")
print("="*80)
for buffer_m, results in buffer_results.items():
    coverage_pct = (results['regions_covered'] / len(scope_regions)) * 100
    print(f"Buffer {buffer_m:3d}m: {results['unique_stations']:4d} stations, "
          f"{results['regions_covered']:5d}/{len(scope_regions):,} regions ({coverage_pct:.1f}%)")


🔍 Performing spatial joins for each buffer distance...

   Buffer: 0m
      ✅ 662 unique stations
      ✅ 662 station-region pairs
      ✅ 201 regions with stations

   Buffer: 50m
      ✅ 2046 unique stations
      ✅ 2046 station-region pairs
      ✅ 518 regions with stations

   Buffer: 100m
      ✅ 3461 unique stations
      ✅ 3461 station-region pairs
      ✅ 858 regions with stations

   Buffer: 200m
      ✅ 6843 unique stations
      ✅ 6843 station-region pairs
      ✅ 1616 regions with stations

SPATIAL JOIN SUMMARY
Buffer   0m:  662 stations,   201/12,130 regions (1.7%)
Buffer  50m: 2046 stations,   518/12,130 regions (4.3%)
Buffer 100m: 3461 stations,   858/12,130 regions (7.1%)
Buffer 200m: 6843 stations,  1616/12,130 regions (13.3%)


## 6. Query Historical Data Availability (2016-2024)

**Note:** Using improved query with `ProcesType: meting` filter for actual measurements only.

In [21]:
def query_historical_data_count(location_code):
    """
    Query historical data count with proper filters.
    Uses ProcesType='meting' to get only actual measurements.
    """
    body = {
        "Locatie": {"Code": location_code},
        "AquoPlusWaarnemingMetadata": {
            "AquoMetadata": {
                "Compartiment": {"Code": "OW"},
                "Grootheid": {"Code": "WATHTE"},
                "ProcesType": "meting"  # Only actual measurements!
            }
        },
        "Periode": {
            "Begindatumtijd": START_DATE,
            "Einddatumtijd": END_DATE
        }
    }
    
    try:
        response = requests.post(
            OBSERVATIONS_ENDPOINT,
            json=body,
            headers=HEADERS,
            timeout=30
        )
        
        if response.status_code == 204:
            return {'count': 0, 'has_data': False, 'error': None}
        
        response.raise_for_status()
        data = response.json()
        
        if "WaarnemingenLijst" in data and data["WaarnemingenLijst"]:
            for obs_series in data["WaarnemingenLijst"]:
                if "MetingenLijst" in obs_series and obs_series["MetingenLijst"]:
                    measurements = obs_series["MetingenLijst"]
                    count = len(measurements)
                    first_ts = measurements[0].get("Tijdstip") if count > 0 else None
                    last_ts = measurements[-1].get("Tijdstip") if count > 0 else None
                    
                    return {
                        'count': count,
                        'has_data': count > 0,
                        'error': None,
                        'first_timestamp': first_ts,
                        'last_timestamp': last_ts
                    }
        
        return {'count': 0, 'has_data': False, 'error': None}
        
    except Exception as e:
        return {'count': 0, 'has_data': False, 'error': str(e)}

print("✅ Historical data query function defined (with ProcesType filter)")

✅ Historical data query function defined (with ProcesType filter)


In [23]:
# Load cache if exists
measurement_cache = {}
if CACHE_FILE.exists():
    print(f"📂 Loading cached results from {CACHE_FILE.name}...")
    with open(CACHE_FILE, 'r') as f:
        measurement_cache = json.load(f)
    print(f"   ✅ Loaded {len(measurement_cache)} cached results")
else:
    print(f"📂 No cache found, will create new one")

# Get unique stations across all buffers  
all_station_codes = set()

# Find location code column in joined data (loc_code_col might be None)
sample_joined = list(buffer_results.values())[0]['joined']
print(f"\n🔍 DEBUG: Columns in joined data: {list(sample_joined.columns)[:15]}")

# Try common column names
for col in ['LOCATIE_CODE', 'locatie_code', 'CODE', 'code']:
    if col in sample_joined.columns:
        loc_code_col = col
        print(f"✅ Using column: '{col}'")
        break

if loc_code_col:
    for buffer_m, results in buffer_results.items():
        all_station_codes.update(results['joined'][loc_code_col].dropna().unique())

print(f"\n🔍 Found {len(all_station_codes)} unique stations across all buffers")
stations_to_query = [code for code in all_station_codes if code not in measurement_cache]
print(f"📡 Need to query: {len(stations_to_query)} stations (others cached)")

# Query API
if stations_to_query:
    print(f"\n⏳ Querying historical data (2016-2024)...\n")
    print(f"   This may take a few minutes (~0.5s per station)\n")
    
    for station_code in tqdm(stations_to_query, desc="Querying stations"):
        result = query_historical_data_count(station_code)
        measurement_cache[station_code] = result
        time.sleep(0.5)
    
    # Save cache
    print(f"\n💾 Saving cache to {CACHE_FILE.name}...")
    with open(CACHE_FILE, 'w') as f:
        json.dump(measurement_cache, f, indent=2)
    print(f"   ✅ Cache saved")
else:
    print(f"\n✅ All stations cached, no queries needed!")

# Statistics
print(f"\n" + "="*80)
print(f"HISTORICAL DATA SUMMARY (2016-2024)")
print(f"="*80)

stations_with_data = sum(1 for v in measurement_cache.values() if v.get('has_data', False))
total_measurements = sum(v.get('count', 0) for v in measurement_cache.values())

print(f"Total stations checked: {len(measurement_cache)}")
print(f"Stations WITH data: {stations_with_data} ({stations_with_data/len(measurement_cache)*100:.1f}%)")
print(f"Total measurements: {total_measurements:,}")
if stations_with_data > 0:
    print(f"Avg measurements per station (with data): {total_measurements/stations_with_data:,.0f}")

📂 No cache found, will create new one

🔍 DEBUG: Columns in joined data: ['FID', 'WAARNEMING_ID', 'NAAM', 'CODE', 'OMSCHRIJVING', 'STATUSWAARDE', 'BEMONSTERINGSHOOGTE', 'REFERENTIEVLAK', 'OPDRACHTGEVENDE_INSTANTIE', 'KWALITEITSWAARDE_CODE', 'WAARDE_LAATSTE_METING', 'TIJDSTIP_LAATSTE_METING', 'PARAMETER_WAT_OMSCHRIJVING', 'BEMONSTERINGSAPPARAATCODE', 'BEMONSTERINGSMETHODECODE']
✅ Using column: 'CODE'

🔍 Found 234 unique stations across all buffers
📡 Need to query: 234 stations (others cached)

⏳ Querying historical data (2016-2024)...

   This may take a few minutes (~0.5s per station)



Querying stations:   0%|          | 0/234 [00:00<?, ?it/s]

Querying stations: 100%|██████████| 234/234 [02:35<00:00,  1.50it/s]


💾 Saving cache to water_stations_historical_cache.json...
   ✅ Cache saved

HISTORICAL DATA SUMMARY (2016-2024)
Total stations checked: 234
Stations WITH data: 1 (0.4%)
Total measurements: 31
Avg measurements per station (with data): 31


In [25]:
# Test: Query recent 1-year period instead
print("🧪 Testing 1-year query (2023-2024) on sample stations...\n")

test_stations = list(all_station_codes)[:10]  # Test first 10
test_results = {}

for station in test_stations:
    body = {
        "Locatie": {"Code": station},
        "AquoPlusWaarnemingMetadata": {
            "AquoMetadata": {
                "Compartiment": {"Code": "OW"},
                "Grootheid": {"Code": "WATHTE"},
                "ProcesType": "meting"
            }
        },
        "Periode": {
            "Begindatumtijd": "2023-01-01T00:00:00.000+01:00",
            "Einddatumtijd": "2024-12-31T23:59:59.000+01:00"
        }
    }
    
    try:
        response = requests.post(OBSERVATIONS_ENDPOINT, json=body, headers=HEADERS, timeout=30)
        if response.status_code != 204:
            data = response.json()
            if "WaarnemingenLijst" in data and data["WaarnemingenLijst"]:
                for obs in data["WaarnemingenLijst"]:
                    if "MetingenLijst" in obs:
                        count = len(obs["MetingenLijst"])
                        test_results[station] = count
                        print(f"   ✅ {station}: {count:,} measurements (2023-2024)")
                        break
    except:
        pass
    time.sleep(0.3)

print(f"\n📊 Results: {len(test_results)}/{len(test_stations)} stations have data for 2023-2024")

🧪 Testing 1-year query (2023-2024) on sample stations...

   ✅ steyl: 105,264 measurements (2023-2024)

📊 Results: 1/10 stations have data for 2023-2024


In [29]:
# Filter to stations with recent data (2015+)
print("🔍 Filtering to stations with recent measurements (2015+)...\n")

active_stations = []
for code in all_station_codes:
    station_info = sample_joined[sample_joined['CODE'] == code]
    if len(station_info) > 0:
        tijdstip = station_info.iloc[0]['TIJDSTIP_LAATSTE_METING']
        try:
            # Parse timestamp
            if pd.notna(tijdstip):
                year = pd.to_datetime(tijdstip).year
                if year >= 2015:
                    active_stations.append(code)
                    print(f"   ✅ {code:40s} Last: {tijdstip}")
        except:
            pass

print(f"\n📊 Active stations (2015+): {len(active_stations)}/{len(all_station_codes)}")
print(f"\n💡 These are the stations we should query for historical data!")

🔍 Filtering to stations with recent measurements (2015+)...

   ✅ steyl                                    Last: 2026-02-04T09:30:00.000Z
   ✅ eisdenmazenhove.maas                     Last: 2026-02-04T09:20:00.000Z
   ✅ culemborg                                Last: 2026-02-04T09:30:00.000Z
   ✅ lith.beneden                             Last: 2026-02-04T09:30:00.000Z
   ✅ arnhem.nederrijn                         Last: 2026-02-04T09:30:00.000Z
   ✅ zwartsluis.zwartewater                   Last: 2026-02-04T09:30:00.000Z
   ✅ zutphen.ijssel                           Last: 2026-02-04T09:30:00.000Z
   ✅ zwolle.ijssel                            Last: 2026-02-04T09:30:00.000Z
   ✅ negenoord.west                           Last: 2026-02-04T09:20:00.000Z
   ✅ lith.boven                               Last: 2026-02-04T09:30:00.000Z
   ✅ kampen.ijssel                            Last: 2026-02-04T09:30:00.000Z
   ✅ sambeek.beneden                          Last: 2026-02-04T09:30:00.000Z
   ✅ krimpenaan

In [30]:
# Check which active stations fall within our buffers
print("🔍 Checking spatial coverage of 64 active stations...\n")

for buffer_m in BUFFER_DISTANCES:
    joined = buffer_results[buffer_m]['joined']
    
    # Filter to active stations only
    active_mask = joined['CODE'].isin(active_stations)
    active_in_buffer = joined[active_mask]
    
    unique_active = active_in_buffer['CODE'].nunique()
    regions_with_active = active_in_buffer[id_col].nunique()
    
    print(f"Buffer {buffer_m:3d}m:")
    print(f"   Active stations: {unique_active}/64 ({unique_active/64*100:.0f}%)")
    print(f"   Regions covered: {regions_with_active}/{len(scope_regions)} ({regions_with_active/len(scope_regions)*100:.1f}%)")
    print()

# Show which active stations are in ANY buffer
all_buffered_active = set()
for buffer_m in BUFFER_DISTANCES:
    all_buffered_active.update(
        buffer_results[buffer_m]['joined'][
            buffer_results[buffer_m]['joined']['CODE'].isin(active_stations)
        ]['CODE'].unique()
    )

print(f"📊 Total active stations in ANY buffer: {len(all_buffered_active)}/64")
print(f"\n🎯 Active stations NEAR your scope regions ({len(all_buffered_active)}):")
for station in sorted(all_buffered_active):
    print(f"   - {station}")

# Stations NOT in buffers
stations_not_in_scope = set(active_stations) - all_buffered_active
if stations_not_in_scope:
    print(f"\n⚠️ Active stations NOT near scope regions ({len(stations_not_in_scope)}):")
    for station in sorted(list(stations_not_in_scope)[:10]):
        print(f"   - {station}")

🔍 Checking spatial coverage of 64 active stations...

Buffer   0m:
   Active stations: 64/64 (100%)
   Regions covered: 63/12130 (0.5%)

Buffer  50m:
   Active stations: 64/64 (100%)
   Regions covered: 213/12130 (1.8%)

Buffer 100m:
   Active stations: 64/64 (100%)
   Regions covered: 340/12130 (2.8%)

Buffer 200m:
   Active stations: 64/64 (100%)
   Regions covered: 608/12130 (5.0%)

📊 Total active stations in ANY buffer: 64/64

🎯 Active stations NEAR your scope regions (64):
   - amerongen.beneden
   - amerongen.boven
   - arnhem.nederrijn
   - buggenum
   - culemborg
   - dalem
   - deventer
   - doesburg.ijssel
   - eisdenmazenhove.maas
   - eisdenmazenhove.maesbempdergreend
   - elsloo.maas
   - genemuiden
   - gennep
   - grave.beneden
   - grave.boven
   - grevenbicht
   - hagestein.beneden
   - hagestein.boven
   - hank.bergschemaas
   - hasselt.zwartewater
   - heel.kanaal
   - heel.maas
   - heesbeen
   - huissen.nederrijn
   - kampen.ijssel
   - kampen.keteldiep
   - krimpe

## 7. Query Historical Data for Active Stations (2016-2025)

Now query the 64 active stations for their full historical data.

In [33]:
def fetch_full_historical_data(location_code, start_date, end_date):
    """
    Fetch complete water level measurements for a station.
    Returns DataFrame with timestamp, water_level_cm, year.
    """
    body = {
        "Locatie": {"Code": location_code},
        "AquoPlusWaarnemingMetadata": {
            "AquoMetadata": {
                "Compartiment": {"Code": "OW"},
                "Grootheid": {"Code": "WATHTE"},
                "ProcesType": "meting"
            }
        },
        "Periode": {
            "Begindatumtijd": start_date,
            "Einddatumtijd": end_date
        }
    }
    
    try:
        response = requests.post(
            OBSERVATIONS_ENDPOINT,
            json=body,
            headers=HEADERS,
            timeout=60
        )
        
        if response.status_code == 204:
            return None
        
        response.raise_for_status()
        data = response.json()
        
        if "WaarnemingenLijst" in data and data["WaarnemingenLijst"]:
            for obs_series in data["WaarnemingenLijst"]:
                if "MetingenLijst" in obs_series and obs_series["MetingenLijst"]:
                    measurements = obs_series["MetingenLijst"]
                    
                    records = []
                    for m in measurements:
                        record = {
                            'timestamp': m.get('Tijdstip'),
                            'water_level_cm': m.get('Meetwaarde', {}).get('Waarde_Numeriek'),
                            'station_code': location_code
                        }
                        records.append(record)
                    
                    df = pd.DataFrame(records)
                    df['timestamp'] = pd.to_datetime(df['timestamp'])
                    df['year'] = df['timestamp'].dt.year
                    return df
        
        return None
        
    except Exception as e:
        return None

print("✅ Full data fetch function defined")

✅ Full data fetch function defined


In [35]:
# Query all 64 active stations for 2016-2025
print(f"📡 Querying {len(active_stations)} active stations for historical data (2016-2025)...\n")
print(f"⏳ This will take ~1-2 minutes (64 stations × ~1 sec each)\n")

all_measurements = []
station_summary = []

for station_code in tqdm(active_stations, desc="Fetching data"):
    df = fetch_full_historical_data(
        station_code,
        "2016-01-01T00:00:00.000+01:00",
        "2025-12-31T23:59:59.000+01:00"
    )
    
    if df is not None and len(df) > 0:
        all_measurements.append(df)
        
        station_summary.append({
            'station_code': station_code,
            'measurement_count': len(df),
            'first_year': df['year'].min(),
            'last_year': df['year'].max(),
            'years_covered': df['year'].nunique(),
            'first_timestamp': df['timestamp'].min(),
            'last_timestamp': df['timestamp'].max()
        })
    else:
        station_summary.append({
            'station_code': station_code,
            'measurement_count': 0,
            'first_year': None,
            'last_year': None,
            'years_covered': 0,
            'first_timestamp': None,
            'last_timestamp': None
        })
    
    time.sleep(0.5)

# Create summary DataFrame
summary_df = pd.DataFrame(station_summary)

print(f"\n" + "="*80)
print(f"HISTORICAL DATA QUERY RESULTS (2016-2025)")
print(f"="*80)
print(f"Stations queried: {len(active_stations)}")
print(f"Stations WITH data: {(summary_df['measurement_count'] > 0).sum()}")
print(f"Stations WITHOUT data: {(summary_df['measurement_count'] == 0).sum()}")
print(f"Total measurements: {summary_df['measurement_count'].sum():,}")

stations_with_data_count = (summary_df['measurement_count'] > 0).sum()
if stations_with_data_count > 0:
    avg_measurements = summary_df[summary_df['measurement_count'] > 0]['measurement_count'].mean()
    print(f"Avg measurements per station (with data): {avg_measurements:,.0f}")

print(f"\n📊 Top 20 stations by measurement count:\n")
print(summary_df.sort_values('measurement_count', ascending=False).head(20).to_string(index=False))

📡 Querying 64 active stations for historical data (2016-2025)...

⏳ This will take ~1-2 minutes (64 stations × ~1 sec each)



Fetching data:   0%|          | 0/64 [00:00<?, ?it/s]Fetching data:   8%|▊         | 5/64 [00:04<00:48,  1.22it/s]Fetching data:  55%|█████▍    | 35/64 [00:23<00:18,  1.59it/s]Fetching data: 100%|██████████| 64/64 [00:56<00:00,  1.14it/s]


HISTORICAL DATA QUERY RESULTS (2016-2025)
Stations queried: 64
Stations WITH data: 4
Stations WITHOUT data: 60
Total measurements: 37,039
Avg measurements per station (with data): 9,260

📊 Top 20 stations by measurement count:

                         station_code  measurement_count  first_year  last_year  years_covered           first_timestamp            last_timestamp
                                rotem               9312      2025.0     2025.0              1 2025-10-28 07:40:00+01:00 2025-12-31 23:50:00+01:00
westervoort.hondbroekschepleij.ijssel               9291      2025.0     2025.0              1 2025-10-28 09:30:00+01:00 2025-12-31 23:50:00+01:00
                           lith.sluis               9281      2025.0     2025.0              1 2025-10-28 09:30:00+01:00 2025-12-31 23:50:00+01:00
                              veessen               9155      2025.0     2025.0              1 2025-10-28 09:30:00+01:00 2025-12-31 23:50:00+01:00
                                 nee

## 8. Visualization: Measurements Per Station (2016-2025)

In [ ]:
# Bar chart: Measurements per station (X-axis = stations)
print("📊 Creating visualization: Measurements per station\n")

# Filter to stations with data
stations_with_data_df = summary_df[summary_df['measurement_count'] > 0].copy()
stations_with_data_df = stations_with_data_df.sort_values('measurement_count', ascending=False)

if len(stations_with_data_df) > 0:
    fig, ax = plt.subplots(figsize=(18, 8))
    
    ax.bar(
        range(len(stations_with_data_df)),
        stations_with_data_df['measurement_count'],
        color='steelblue',
        edgecolor='black',
        alpha=0.8
    )
    
    ax.set_xlabel('Station', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Measurements (2016-2025)', fontsize=12, fontweight='bold')
    ax.set_title(f'Water Level Measurements by Station ({len(stations_with_data_df)} Stations with Data)', 
                 fontsize=14, fontweight='bold')
    
    # Set x-axis labels (station codes)
    ax.set_xticks(range(len(stations_with_data_df)))
    ax.set_xticklabels(
        stations_with_data_df['station_code'],
        rotation=90,
        ha='right',
        fontsize=8
    )
    
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 Insights:")
    print(f"   • Top station: {stations_with_data_df.iloc[0]['station_code']} "
          f"({stations_with_data_df.iloc[0]['measurement_count']:,} measurements)")
    print(f"   • Median: {stations_with_data_df['measurement_count'].median():,.0f} measurements")
    print(f"   • Range: {stations_with_data_df['measurement_count'].min():,} to "
          f"{stations_with_data_df['measurement_count'].max():,}")
else:
    print("⚠️ No stations with data to visualize")

## 9. Visualization: Measurements Per Year (2016-2025)

In [ ]:
# Bar chart: Measurements per year (X-axis = years)
print("📊 Creating visualization: Measurements per year\n")

if all_measurements:
    # Combine all measurements from all stations
    combined_df = pd.concat(all_measurements, ignore_index=True)
    
    # Count measurements per year
    yearly_counts = combined_df.groupby('year').size().reset_index(name='count')
    yearly_counts = yearly_counts.sort_values('year')
    
    # Filter to 2016-2025
    yearly_counts = yearly_counts[(yearly_counts['year'] >= 2016) & (yearly_counts['year'] <= 2025)]
    
    print(f"📊 Yearly measurement summary:\n")
    print(yearly_counts.to_string(index=False))
    
    # Count unique stations per year
    stations_per_year = combined_df.groupby('year')['station_code'].nunique().reset_index(name='stations')
    yearly_counts = yearly_counts.merge(stations_per_year, on='year')
    
    print(f"\n📊 With station counts:\n")
    print(yearly_counts.to_string(index=False))
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(14, 7))
    
    ax.bar(
        yearly_counts['year'],
        yearly_counts['count'],
        color='darkgreen',
        edgecolor='black',
        alpha=0.8,
        width=0.7
    )
    
    # Add station count as text on bars
    for idx, row in yearly_counts.iterrows():
        ax.text(
            row['year'],
            row['count'] + yearly_counts['count'].max() * 0.02,
            f"{row['stations']} stations",
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )
    
    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('Total Measurements (All Stations)', fontsize=12, fontweight='bold')
    ax.set_title('Water Level Measurements Per Year (2016-2025)', 
                 fontsize=14, fontweight='bold')
    
    ax.set_xticks(yearly_counts['year'])
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 Insights:")
    print(f"   • Total measurements (2016-2025): {yearly_counts['count'].sum():,}")
    peak_year = yearly_counts.loc[yearly_counts['count'].idxmax()]
    print(f"   • Peak year: {int(peak_year['year'])} ({int(peak_year['count']):,} measurements, {int(peak_year['stations'])} stations)")
    print(f"   • Avg per year: {yearly_counts['count'].mean():,.0f}")
else:
    print("⚠️ No measurement data to visualize")

## 10. Save to GeoPackage

Save scope regions and active water stations to the output GeoPackage.

In [ ]:
print(f"📦 Creating output GeoPackage: {OUTPUT_GPKG.name}\n")

# Step 1: Copy scope regions
print(f"   Step 1: Copying scope regions...")
scope_regions.to_file(OUTPUT_GPKG, layer=SCOPE_LAYER, driver='GPKG')
print(f"      ✅ Saved layer: {SCOPE_LAYER}")

# Step 2: Save active stations for each buffer
print(f"\n   Step 2: Saving active water station layers...")

for buffer_m in BUFFER_DISTANCES:
    joined = buffer_results[buffer_m]['joined']
    
    # Filter to ACTIVE stations only (2015+)
    active_joined = joined[joined['CODE'].isin(active_stations)].copy()
    
    # Add measurement summary metadata
    active_joined = active_joined.merge(
        summary_df[['station_code', 'measurement_count', 'first_year', 'last_year', 'years_covered']],
        left_on='CODE',
        right_on='station_code',
        how='left'
    )
    
    layer_name = f"water_stations_{buffer_m}m"
    
    # Save
    active_joined.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG')
    
    # Count stations with historical data
    with_historical = (active_joined['measurement_count'].fillna(0) > 0).sum()
    unique_stations = active_joined['CODE'].nunique()
    
    print(f"      ✅ Saved: {layer_name}")
    print(f"         {unique_stations} active stations, {with_historical} with 2016-2025 data")

print(f"\n✅ GeoPackage saved successfully!")
print(f"\n📂 Output: {OUTPUT_GPKG}")

## 11. Summary & Next Steps

In [ ]:
print("="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\n📊 Data processed:")
print(f"   • {len(scope_regions):,} scope regions")
print(f"   • 64 active water level stations (2015+)")
print(f"   • {stations_with_data_count} stations with historical data (2016-2025)")
if stations_with_data_count > 0:
    print(f"   • {summary_df['measurement_count'].sum():,} total measurements")

print(f"\n📦 Output: {OUTPUT_GPKG.name}")
print(f"   Layers:")
print(f"   • {SCOPE_LAYER} ({len(scope_regions):,} regions)")
for buffer_m in BUFFER_DISTANCES:
    active_mask = buffer_results[buffer_m]['joined']['CODE'].isin(active_stations)
    regions_covered = buffer_results[buffer_m]['joined'][active_mask][id_col].nunique()
    unique_stations = buffer_results[buffer_m]['joined'][active_mask]['CODE'].nunique()
    print(f"   • water_stations_{buffer_m}m ({unique_stations} active stations, {regions_covered:,} regions covered)")

print(f"\n🎯 Coverage (active stations with 100m buffer):")
active_100m = buffer_results[100]['joined'][buffer_results[100]['joined']['CODE'].isin(active_stations)]
regions_100m = active_100m[id_col].nunique()
coverage_100m = (regions_100m / len(scope_regions)) * 100
print(f"   • {regions_100m:,}/{len(scope_regions):,} regions have a water station within 100m ({coverage_100m:.1f}%)")

print(f"\n💡 Key findings:")
print(f"   • WFS service filtered 840k+ monitoring points → 2,599 water height stations")
print(f"   • Only 64 stations are currently active (2015+)")
print(f"   • All 64 active stations are within 200m of your scope regions")
if stations_with_data_count > 0:
    print(f"   • {stations_with_data_count} stations returned historical data via API")
    print(f"   • Average {avg_measurements:,.0f} measurements per active station")

print(f"\n🚀 Next steps:")
print(f"   1. Open {OUTPUT_GPKG.name} in QGIS")
print(f"      - Visualize station coverage by buffer")
print(f"      - Check which regions have nearby stations")
print(f"   2. Decide on buffer distance (100m recommended)")
print(f"   3. Implement in DataHandler:")
print(f"      - Load water_stations layer from GeoPackage")
print(f"      - Match stations to scope regions")
print(f"      - Extract high water events (e.g., >95th percentile)")
print(f"      - Create features: max_water_level, high_water_frequency, etc.")
print(f"   4. Handle regions without stations:")
print(f"      - Use nearest neighbor interpolation")
print(f"      - Or use normalized/zero values")

print(f"\n" + "="*80)

## DEBUG: Test with Known-Good Station

Before accepting the results, let's verify the query works with `steyl` (which we know has historical data).

In [36]:
# Test with steyl - we know this has data
print("🧪 Testing query function with 'steyl' (known-good station)...\n")

test_df = fetch_full_historical_data(
    "steyl",
    "2023-01-01T00:00:00.000+01:00",
    "2024-12-31T23:59:59.000+01:00"
)

if test_df is not None:
    print(f"✅ SUCCESS - Retrieved {len(test_df):,} measurements")
    print(f"   Year range: {test_df['year'].min()}-{test_df['year'].max()}")
    print(f"   Date range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
    print(f"\n📊 Sample data:")
    print(test_df.head(10))
else:
    print("❌ FAILED - No data returned")
    print("   This means our query function has a problem!")

# Also check if 'steyl' is in our active stations list
print(f"\n🔍 Is 'steyl' in our 64 active stations? {'steyl' in active_stations}")

🧪 Testing query function with 'steyl' (known-good station)...

✅ SUCCESS - Retrieved 105,264 measurements
   Year range: 2023-2024
   Date range: 2023-01-01 00:00:00+01:00 to 2024-12-31 23:50:00+01:00

📊 Sample data:
                  timestamp  water_level_cm station_code  year
0 2023-01-01 00:00:00+01:00          1181.0        steyl  2023
1 2023-01-01 00:10:00+01:00          1181.0        steyl  2023
2 2023-01-01 00:20:00+01:00          1181.0        steyl  2023
3 2023-01-01 00:30:00+01:00          1181.0        steyl  2023
4 2023-01-01 00:40:00+01:00          1181.0        steyl  2023
5 2023-01-01 00:50:00+01:00          1182.0        steyl  2023
6 2023-01-01 01:00:00+01:00          1182.0        steyl  2023
7 2023-01-01 01:10:00+01:00          1182.0        steyl  2023
8 2023-01-01 01:20:00+01:00          1182.0        steyl  2023
9 2023-01-01 01:30:00+01:00          1182.0        steyl  2023

🔍 Is 'steyl' in our 64 active stations? True


In [ ]:
# Test with a station that returned NO data - investigate the API response
print("🔍 Testing a station that returned no data: 'deventer'\n")

body = {
    "Locatie": {"Code": "deventer"},
    "AquoPlusWaarnemingMetadata": {
        "AquoMetadata": {
            "Compartiment": {"Code": "OW"},
            "Grootheid": {"Code": "WATHTE"},
            "ProcesType": "meting"
        }
    },
    "Periode": {
        "Begindatumtijd": "2023-01-01T00:00:00.000+01:00",
        "Einddatumtijd": "2024-12-31T23:59:59.000+01:00"
    }
}

response = requests.post(
    OBSERVATIONS_ENDPOINT,
    json=body,
    headers=HEADERS,
    timeout=60
)

print(f"Status code: {response.status_code}")

if response.status_code == 204:
    print("   → 204 No Content (station has no data for this period)")
elif response.status_code == 200:
    data = response.json()
    print(f"   → 200 OK (data returned)")
    print(f"\n📦 Response structure:")
    print(f"   Keys: {list(data.keys())}")
    
    if "WaarnemingenLijst" in data:
        print(f"   WaarnemingenLijst length: {len(data['WaarnemingenLijst'])}")
        if data['WaarnemingenLijst']:
            first_obs = data['WaarnemingenLijst'][0]
            print(f"   First observation keys: {list(first_obs.keys())}")
            if "MetingenLijst" in first_obs:
                print(f"   MetingenLijst length: {len(first_obs['MetingenLijst'])}")
            else:
                print(f"   ⚠️ No 'MetingenLijst' in observation!")
    else:
        print(f"   ⚠️ No 'WaarnemingenLijst' in response!")
else:
    print(f"   → Error: {response.status_code}")
    print(f"   {response.text[:500]}")

🔍 Testing a station that returned no data: 'deventer'

Status code: 200
   → 200 OK (data returned)

📦 Response structure:
   Keys: ['Succesvol', 'WaarnemingenLijst']
   WaarnemingenLijst length: 1
   First observation keys: ['AquoMetadata', 'Locatie', 'MetingenLijst']
   MetingenLijst length: 105216


In [ ]:
# Analyze the 4 stations that DID return data
print("📊 Analyzing the 4 stations that returned data:\n")

stations_with_data_df = summary_df[summary_df['measurement_count'] > 0].copy()
print(stations_with_data_df[['station_code', 'measurement_count', 'first_year', 'last_year', 'years_covered']].to_string(index=False))

print(f"\n💡 Pattern detected:")
print(f"   • All 4 stations have data ONLY for 2025")
print(f"   • First timestamp: October 28, 2025")
print(f"   • This suggests these stations are newly activated or recently digitized")

print(f"\n❓ Questions:")
print(f"   1. Why do these specific 4 stations have 2025 data but not 2016-2024?")
print(f"   2. Why do the other 60 stations have NO data at all via the API?")
print(f"   3. Is the API only providing very recent data?")

print(f"\n🔍 Let's check the WFS metadata for these 4 stations:")
for station_code in stations_with_data_df['station_code']:
    station_info = water_stations_gdf[water_stations_gdf['CODE'] == station_code]
    if len(station_info) > 0:
        latest = station_info['TIJDSTIP_LAATSTE_METING'].iloc[0]
        print(f"   • {station_code}: WFS says latest measurement = {latest}")

📊 Analyzing the 4 stations that returned data:

                         station_code  measurement_count  first_year  last_year  years_covered
                                rotem               9312      2025.0     2025.0              1
                              veessen               9155      2025.0     2025.0              1
westervoort.hondbroekschepleij.ijssel               9291      2025.0     2025.0              1
                           lith.sluis               9281      2025.0     2025.0              1

💡 Pattern detected:
   • All 4 stations have data ONLY for 2025
   • First timestamp: October 28, 2025
   • This suggests these stations are newly activated or recently digitized

❓ Questions:
   1. Why do these specific 4 stations have 2025 data but not 2016-2024?
   2. Why do the other 60 stations have NO data at all via the API?
   3. Is the API only providing very recent data?

🔍 Let's check the WFS metadata for these 4 stations:
   • rotem: WFS says latest measuremen

## DEBUG: Find the Parsing Bug

The API returns data for `deventer` but our function returns None. Let's trace through the parsing step-by-step.

In [39]:
# Fetch deventer data again and trace through parsing
print("🔍 Detailed parsing trace for 'deventer'...\n")

body = {
    "Locatie": {"Code": "deventer"},
    "AquoPlusWaarnemingMetadata": {
        "AquoMetadata": {
            "Compartiment": {"Code": "OW"},
            "Grootheid": {"Code": "WATHTE"},
            "ProcesType": "meting"
        }
    },
    "Periode": {
        "Begindatumtijd": "2023-01-01T00:00:00.000+01:00",
        "Einddatumtijd": "2024-12-31T23:59:59.000+01:00"
    }
}

response = requests.post(OBSERVATIONS_ENDPOINT, json=body, headers=HEADERS, timeout=60)
data = response.json()

print(f"✅ API Response received")
print(f"   WaarnemingenLijst entries: {len(data['WaarnemingenLijst'])}")

for idx, obs_series in enumerate(data['WaarnemingenLijst']):
    print(f"\n   Processing observation series {idx}:")
    print(f"      Keys: {list(obs_series.keys())}")
    
    if "MetingenLijst" in obs_series:
        print(f"      ✅ Has MetingenLijst: {len(obs_series['MetingenLijst'])} measurements")
        
        # Try to parse first few measurements
        print(f"      🔍 Parsing first measurement...")
        try:
            first_m = obs_series['MetingenLijst'][0]
            print(f"         Keys: {list(first_m.keys())}")
            print(f"         Tijdstip: {first_m.get('Tijdstip')}")
            print(f"         Meetwaarde: {first_m.get('Meetwaarde')}")
            
            if first_m.get('Meetwaarde'):
                print(f"         Meetwaarde type: {type(first_m.get('Meetwaarde'))}")
                print(f"         Meetwaarde keys: {list(first_m.get('Meetwaarde').keys()) if isinstance(first_m.get('Meetwaarde'), dict) else 'Not a dict'}")
                print(f"         Waarde_Numeriek: {first_m.get('Meetwaarde', {}).get('Waarde_Numeriek')}")
            
            # Try to create a record
            record = {
                'timestamp': first_m.get('Tijdstip'),
                'water_level_cm': first_m.get('Meetwaarde', {}).get('Waarde_Numeriek'),
                'station_code': 'deventer'
            }
            print(f"         ✅ Record created: {record}")
            
            # Try to create DataFrame from a few records
            records = []
            for m in obs_series['MetingenLijst'][:10]:
                records.append({
                    'timestamp': m.get('Tijdstip'),
                    'water_level_cm': m.get('Meetwaarde', {}).get('Waarde_Numeriek'),
                    'station_code': 'deventer'
                })
            
            test_df = pd.DataFrame(records)
            print(f"         ✅ DataFrame created with {len(test_df)} rows")
            print(f"         Columns: {list(test_df.columns)}")
            
            # Try datetime conversion
            test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])
            print(f"         ✅ Timestamp conversion successful")
            
            # Try year extraction
            test_df['year'] = test_df['timestamp'].dt.year
            print(f"         ✅ Year extraction successful")
            print(f"         Years: {test_df['year'].unique()}")
            
        except Exception as e:
            print(f"         ❌ ERROR: {type(e).__name__}: {e}")
            import traceback
            print(f"         {traceback.format_exc()}")
    else:
        print(f"      ❌ No MetingenLijst!")

print("\n💡 If all steps succeeded, our function should work. Let's test it again:")

🔍 Detailed parsing trace for 'deventer'...

✅ API Response received
   WaarnemingenLijst entries: 1

   Processing observation series 0:
      Keys: ['AquoMetadata', 'Locatie', 'MetingenLijst']
      ✅ Has MetingenLijst: 105216 measurements
      🔍 Parsing first measurement...
         Keys: ['Meetwaarde', 'Tijdstip', 'WaarnemingMetadata']
         Tijdstip: 2023-01-01T00:00:00.000+01:00
         Meetwaarde: {'Waarde_Alfanumeriek': '427', 'Waarde_Numeriek': 427.0}
         Meetwaarde type: <class 'dict'>
         Meetwaarde keys: ['Waarde_Alfanumeriek', 'Waarde_Numeriek']
         Waarde_Numeriek: 427.0
         ✅ Record created: {'timestamp': '2023-01-01T00:00:00.000+01:00', 'water_level_cm': 427.0, 'station_code': 'deventer'}
         ✅ DataFrame created with 10 rows
         Columns: ['timestamp', 'water_level_cm', 'station_code']
         ✅ Timestamp conversion successful
         ✅ Year extraction successful
         Years: [2023]

💡 If all steps succeeded, our function should wor

In [40]:
# Test our actual function with deventer
print("🧪 Testing fetch_full_historical_data() with 'deventer'...\n")

result = fetch_full_historical_data(
    "deventer",
    "2023-01-01T00:00:00.000+01:00",
    "2024-12-31T23:59:59.000+01:00"
)

if result is not None:
    print(f"✅ SUCCESS! Retrieved {len(result):,} measurements")
    print(result.head())
else:
    print(f"❌ FAILED - Function returned None")
    print(f"\n💡 The parsing trace above should show us why!")

🧪 Testing fetch_full_historical_data() with 'deventer'...

✅ SUCCESS! Retrieved 105,216 measurements
                  timestamp  water_level_cm station_code  year
0 2023-01-01 00:00:00+01:00           427.0     deventer  2023
1 2023-01-01 00:10:00+01:00           427.0     deventer  2023
2 2023-01-01 00:20:00+01:00           426.0     deventer  2023
3 2023-01-01 00:30:00+01:00           427.0     deventer  2023
4 2023-01-01 00:40:00+01:00           427.0     deventer  2023


In [41]:
# HYPOTHESIS: Maybe the 2016-2025 date range is the problem?
print("🔍 Testing different date ranges for 'deventer'...\n")

date_ranges = [
    ("2023-2024", "2023-01-01T00:00:00.000+01:00", "2024-12-31T23:59:59.000+01:00"),
    ("2016-2025", "2016-01-01T00:00:00.000+01:00", "2025-12-31T23:59:59.000+01:00"),
    ("2020-2024", "2020-01-01T00:00:00.000+01:00", "2024-12-31T23:59:59.000+01:00"),
]

for label, start, end in date_ranges:
    result = fetch_full_historical_data("deventer", start, end)
    if result is not None:
        print(f"   {label}: ✅ {len(result):,} measurements (years: {result['year'].min()}-{result['year'].max()})")
    else:
        print(f"   {label}: ❌ No data")

🔍 Testing different date ranges for 'deventer'...

   2023-2024: ✅ 105,216 measurements (years: 2023-2024)
   2016-2025: ❌ No data
   2020-2024: ❌ No data


## SOLUTION: Chunked Data Collection (Year-by-Year)

The API has a volume limit. Solution: Query year-by-year and concatenate results.

In [42]:
def fetch_historical_data_chunked(location_code, start_year=2016, end_year=2025):
    """
    Fetch water level data year-by-year to avoid API volume limits.
    Returns combined DataFrame with all measurements.
    """
    all_dfs = []
    
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01T00:00:00.000+01:00"
        end_date = f"{year}-12-31T23:59:59.000+01:00"
        
        body = {
            "Locatie": {"Code": location_code},
            "AquoPlusWaarnemingMetadata": {
                "AquoMetadata": {
                    "Compartiment": {"Code": "OW"},
                    "Grootheid": {"Code": "WATHTE"},
                    "ProcesType": "meting"
                }
            },
            "Periode": {
                "Begindatumtijd": start_date,
                "Einddatumtijd": end_date
            }
        }
        
        try:
            response = requests.post(
                OBSERVATIONS_ENDPOINT,
                json=body,
                headers=HEADERS,
                timeout=60
            )
            
            if response.status_code == 204:
                continue  # No data for this year
            
            response.raise_for_status()
            data = response.json()
            
            if "WaarnemingenLijst" in data and data["WaarnemingenLijst"]:
                for obs_series in data["WaarnemingenLijst"]:
                    if "MetingenLijst" in obs_series and obs_series["MetingenLijst"]:
                        measurements = obs_series["MetingenLijst"]
                        
                        records = []
                        for m in measurements:
                            record = {
                                'timestamp': m.get('Tijdstip'),
                                'water_level_cm': m.get('Meetwaarde', {}).get('Waarde_Numeriek'),
                                'station_code': location_code
                            }
                            records.append(record)
                        
                        if records:
                            df = pd.DataFrame(records)
                            df['timestamp'] = pd.to_datetime(df['timestamp'])
                            df['year'] = df['timestamp'].dt.year
                            all_dfs.append(df)
                            
        except Exception as e:
            # Continue with other years even if one fails
            continue
        
        # Small delay to be nice to the API
        time.sleep(0.2)
    
    # Combine all years
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        return combined_df
    else:
        return None

print("✅ Chunked fetch function defined (queries year-by-year)")

✅ Chunked fetch function defined (queries year-by-year)


In [ ]:
# Test chunked approach with deventer
print("🧪 Testing chunked fetch on 'deventer' (2016-2025)...\n")
print("   This will query 10 years individually (should take ~5 seconds)\n")

result = fetch_historical_data_chunked("deventer", start_year=2016, end_year=2025)

if result is not None:
    print(f"✅ SUCCESS! Retrieved {len(result):,} measurements")
    print(f"   Year range: {result['year'].min()}-{result['year'].max()}")
    print(f"   Years covered: {sorted(result['year'].unique())}")
    print(f"   Measurements per year:")
    
    yearly_counts = result.groupby('year').size().sort_index()
    for year, count in yearly_counts.items():
        print(f"      {year}: {count:,}")
    
    print(f"\n📊 Sample data:")
    print(result.head(10))
else:
    print(f"❌ FAILED - No data returned")

🧪 Testing chunked fetch on 'deventer' (2016-2025)...

   This will query 10 years individually (should take ~5 seconds)



In [44]:
# Test with steyl too
print("🧪 Testing chunked fetch on 'steyl' (2016-2025)...\n")

result_steyl = fetch_historical_data_chunked("steyl", start_year=2016, end_year=2025)

if result_steyl is not None:
    print(f"✅ SUCCESS! Retrieved {len(result_steyl):,} measurements")
    print(f"   Year range: {result_steyl['year'].min()}-{result_steyl['year'].max()}")
    print(f"   Years covered: {sorted(result_steyl['year'].unique())}")
    
    yearly_counts = result_steyl.groupby('year').size().sort_index()
    print(f"\n   Measurements per year:")
    for year, count in yearly_counts.items():
        print(f"      {year}: {count:,}")
else:
    print(f"❌ FAILED - No data returned")

🧪 Testing chunked fetch on 'steyl' (2016-2025)...

✅ SUCCESS! Retrieved 533,214 measurements
   Year range: 2016-2025
   Years covered: [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]

   Measurements per year:
      2016: 53,436
      2017: 53,290
      2018: 53,290
      2019: 52,925
      2020: 53,070
      2021: 52,560
      2022: 52,560
      2023: 52,560
      2024: 52,704
      2025: 56,819


## Re-run Full Query with Chunked Approach

Now query all 64 active stations using the year-by-year method.

In [46]:
# Query all 64 active stations with chunked approach
print(f"📡 Querying {len(active_stations)} active stations (CHUNKED: year-by-year)...\n")
print(f"⏳ This will take ~5-10 minutes (64 stations × 10 years × ~0.5s)\n")
print(f"💡 Each station queries 10 years separately to avoid API limits\n")

all_measurements_chunked = []
station_summary_chunked = []

for station_code in tqdm(active_stations, desc="Fetching data (chunked)"):
    df = fetch_historical_data_chunked(
        station_code,
        start_year=2016,
        end_year=2025
    )
    
    if df is not None and len(df) > 0:
        all_measurements_chunked.append(df)
        
        station_summary_chunked.append({
            'station_code': station_code,
            'measurement_count': len(df),
            'first_year': df['year'].min(),
            'last_year': df['year'].max(),
            'years_covered': df['year'].nunique(),
            'first_timestamp': df['timestamp'].min(),
            'last_timestamp': df['timestamp'].max()
        })
    else:
        station_summary_chunked.append({
            'station_code': station_code,
            'measurement_count': 0,
            'first_year': None,
            'last_year': None,
            'years_covered': 0,
            'first_timestamp': None,
            'last_timestamp': None
        })

# Create summary DataFrame
summary_df_chunked = pd.DataFrame(station_summary_chunked)

print(f"\n" + "="*80)
print(f"CHUNKED QUERY RESULTS (2016-2025)")
print(f"="*80)
print(f"Stations queried: {len(active_stations)}")
print(f"Stations WITH data: {(summary_df_chunked['measurement_count'] > 0).sum()}")
print(f"Stations WITHOUT data: {(summary_df_chunked['measurement_count'] == 0).sum()}")
print(f"Total measurements: {summary_df_chunked['measurement_count'].sum():,}")

stations_with_data_count_chunked = (summary_df_chunked['measurement_count'] > 0).sum()
if stations_with_data_count_chunked > 0:
    avg_measurements_chunked = summary_df_chunked[summary_df_chunked['measurement_count'] > 0]['measurement_count'].mean()
    print(f"Avg measurements per station (with data): {avg_measurements_chunked:,.0f}")

print(f"\n📊 Top 20 stations by measurement count:\n")
print(summary_df_chunked.sort_values('measurement_count', ascending=False).head(20).to_string(index=False))

📡 Querying 64 active stations (CHUNKED: year-by-year)...

⏳ This will take ~5-10 minutes (64 stations × 10 years × ~0.5s)

💡 Each station queries 10 years separately to avoid API limits



Fetching data (chunked):   3%|▎         | 2/64 [01:40<50:54, 49.26s/it]Fetching data (chunked):   5%|▍         | 3/64 [02:41<54:36, 53.72s/it]


KeyboardInterrupt: 

## Visualization: Measurements Per Station (CHUNKED DATA)

In [ ]:
# Bar chart: Measurements per station (X-axis = stations) - CHUNKED DATA
print("📊 Creating visualization: Measurements per station (2016-2025)\n")

# Filter to stations with data
stations_with_data_df_chunked = summary_df_chunked[summary_df_chunked['measurement_count'] > 0].copy()
stations_with_data_df_chunked = stations_with_data_df_chunked.sort_values('measurement_count', ascending=False)

if len(stations_with_data_df_chunked) > 0:
    fig, ax = plt.subplots(figsize=(18, 8))
    
    ax.bar(
        range(len(stations_with_data_df_chunked)),
        stations_with_data_df_chunked['measurement_count'],
        color='steelblue',
        edgecolor='black',
        alpha=0.8
    )
    
    ax.set_xlabel('Station', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Measurements (2016-2025)', fontsize=12, fontweight='bold')
    ax.set_title(f'Water Level Measurements by Station ({len(stations_with_data_df_chunked)} Stations with Data)', 
                 fontsize=14, fontweight='bold')
    
    # Set x-axis labels (station codes)
    ax.set_xticks(range(len(stations_with_data_df_chunked)))
    ax.set_xticklabels(
        stations_with_data_df_chunked['station_code'],
        rotation=90,
        ha='right',
        fontsize=8
    )
    
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 Insights:")
    print(f"   • Top station: {stations_with_data_df_chunked.iloc[0]['station_code']} "
          f"({stations_with_data_df_chunked.iloc[0]['measurement_count']:,} measurements)")
    print(f"   • Median: {stations_with_data_df_chunked['measurement_count'].median():,.0f} measurements")
    print(f"   • Range: {stations_with_data_df_chunked['measurement_count'].min():,} to "
          f"{stations_with_data_df_chunked['measurement_count'].max():,}")
else:
    print("⚠️ No stations with data to visualize")

## Visualization: Measurements Per Year (CHUNKED DATA)

In [ ]:
# Bar chart: Measurements per year (X-axis = years) - CHUNKED DATA
print("📊 Creating visualization: Measurements per year (2016-2025)\n")

if all_measurements_chunked:
    # Combine all measurements from all stations
    combined_df_chunked = pd.concat(all_measurements_chunked, ignore_index=True)
    
    # Count measurements per year
    yearly_counts_chunked = combined_df_chunked.groupby('year').size().reset_index(name='count')
    yearly_counts_chunked = yearly_counts_chunked.sort_values('year')
    
    # Filter to 2016-2025
    yearly_counts_chunked = yearly_counts_chunked[(yearly_counts_chunked['year'] >= 2016) & (yearly_counts_chunked['year'] <= 2025)]
    
    print(f"📊 Yearly measurement summary:\n")
    print(yearly_counts_chunked.to_string(index=False))
    
    # Count unique stations per year
    stations_per_year_chunked = combined_df_chunked.groupby('year')['station_code'].nunique().reset_index(name='stations')
    yearly_counts_chunked = yearly_counts_chunked.merge(stations_per_year_chunked, on='year')
    
    print(f"\n📊 With station counts:\n")
    print(yearly_counts_chunked.to_string(index=False))
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(14, 7))
    
    ax.bar(
        yearly_counts_chunked['year'],
        yearly_counts_chunked['count'],
        color='darkgreen',
        edgecolor='black',
        alpha=0.8,
        width=0.7
    )
    
    # Add station count as text on bars
    for idx, row in yearly_counts_chunked.iterrows():
        ax.text(
            row['year'],
            row['count'] + yearly_counts_chunked['count'].max() * 0.02,
            f"{row['stations']} stations",
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )
    
    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('Total Measurements (All Stations)', fontsize=12, fontweight='bold')
    ax.set_title('Water Level Measurements Per Year (2016-2025)', 
                 fontsize=14, fontweight='bold')
    
    ax.set_xticks(yearly_counts_chunked['year'])
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 Insights:")
    print(f"   • Total measurements (2016-2025): {yearly_counts_chunked['count'].sum():,}")
    peak_year = yearly_counts_chunked.loc[yearly_counts_chunked['count'].idxmax()]
    print(f"   • Peak year: {int(peak_year['year'])} ({int(peak_year['count']):,} measurements, {int(peak_year['stations'])} stations)")
    print(f"   • Avg per year: {yearly_counts_chunked['count'].mean():,.0f}")
else:
    print("⚠️ No measurement data to visualize")

## Save to GeoPackage (FINAL - with Chunked Data)

In [ ]:
print(f"📦 Creating output GeoPackage: {OUTPUT_GPKG.name}\n")

# Step 1: Copy scope regions
print(f"   Step 1: Copying scope regions...")
scope_regions.to_file(OUTPUT_GPKG, layer=SCOPE_LAYER, driver='GPKG')
print(f"      ✅ Saved layer: {SCOPE_LAYER}")

# Step 2: Save active stations for each buffer with CHUNKED summary data
print(f"\n   Step 2: Saving active water station layers with historical data...")

for buffer_m in BUFFER_DISTANCES:
    joined = buffer_results[buffer_m]['joined']
    
    # Filter to ACTIVE stations only (2015+)
    active_joined = joined[joined['CODE'].isin(active_stations)].copy()
    
    # Add CHUNKED measurement summary metadata
    active_joined = active_joined.merge(
        summary_df_chunked[['station_code', 'measurement_count', 'first_year', 'last_year', 'years_covered']],
        left_on='CODE',
        right_on='station_code',
        how='left'
    )
    
    layer_name = f"water_stations_{buffer_m}m"
    
    # Save
    active_joined.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG')
    
    # Count stations with historical data
    with_historical = (active_joined['measurement_count'].fillna(0) > 0).sum()
    unique_stations = active_joined['CODE'].nunique()
    
    print(f"      ✅ Saved: {layer_name}")
    print(f"         {unique_stations} active stations, {with_historical} with 2016-2025 data")

print(f"\n✅ GeoPackage saved successfully!")
print(f"\n📂 Output: {OUTPUT_GPKG}")

## Final Summary & Next Steps

In [ ]:
print("="*80)
print("FINAL SUMMARY (CHUNKED DATA COLLECTION)")
print("="*80)

print(f"\n📊 Data processed:")
print(f"   • {len(scope_regions):,} scope regions")
print(f"   • 64 active water level stations (2015+)")
print(f"   • {stations_with_data_count_chunked} stations with historical data (2016-2025)")
if stations_with_data_count_chunked > 0:
    print(f"   • {summary_df_chunked['measurement_count'].sum():,} total measurements")

print(f"\n📦 Output: {OUTPUT_GPKG.name}")
print(f"   Layers:")
print(f"   • {SCOPE_LAYER} ({len(scope_regions):,} regions)")
for buffer_m in BUFFER_DISTANCES:
    active_mask = buffer_results[buffer_m]['joined']['CODE'].isin(active_stations)
    regions_covered = buffer_results[buffer_m]['joined'][active_mask][id_col].nunique()
    unique_stations = buffer_results[buffer_m]['joined'][active_mask]['CODE'].nunique()
    print(f"   • water_stations_{buffer_m}m ({unique_stations} active stations, {regions_covered:,} regions covered)")

print(f"\n🎯 Coverage (active stations with 100m buffer):")
active_100m = buffer_results[100]['joined'][buffer_results[100]['joined']['CODE'].isin(active_stations)]
regions_100m = active_100m[id_col].nunique()
coverage_100m = (regions_100m / len(scope_regions)) * 100
print(f"   • {regions_100m:,}/{len(scope_regions):,} regions have a water station within 100m ({coverage_100m:.1f}%)")

print(f"\n💡 Key findings:")
print(f"   • WFS service filtered 840k+ monitoring points → 2,599 water height stations")
print(f"   • Only 64 stations are currently active (2015+)")
print(f"   • All 64 active stations are within 200m of your scope regions")
if stations_with_data_count_chunked > 0:
    print(f"   • {stations_with_data_count_chunked} stations have historical data via API (year-by-year query)")
    print(f"   • Average {avg_measurements_chunked:,.0f} measurements per active station")
    print(f"   • API has volume limits - chunked queries (year-by-year) are required!")

print(f"\n🔧 Technical lessons:")
print(f"   • API rejects requests for >2 years of data (returns 204/empty)")
print(f"   • Solution: Query year-by-year and concatenate")
print(f"   • Similar to DataCollector's chunking strategy")

print(f"\n🚀 Next steps:")
print(f"   1. Open {OUTPUT_GPKG.name} in QGIS")
print(f"      - Visualize station coverage by buffer")
print(f"      - Check which regions have nearby stations")
print(f"      - Color by 'measurement_count' field to see data quality")
print(f"   2. Decide on buffer distance (100m recommended)")
print(f"   3. Implement in DataHandler:")
print(f"      - Load water_stations layer from GeoPackage")
print(f"      - Match stations to scope regions")
print(f"      - Use chunked API queries to fetch time series")
print(f"      - Extract high water events (e.g., >95th percentile)")
print(f"      - Create features: max_water_level, high_water_frequency, etc.")
print(f"   4. Handle regions without stations:")
print(f"      - Use nearest neighbor interpolation")
print(f"      - Or use normalized/zero values")

print(f"\n" + "="*80)

## QUICK EXPORT: Save Active Stations Without Time Series

Skip the lengthy time series query and just export the 64 active station locations with metadata.

In [49]:
print(f"📦 QUICK EXPORT: Saving 64 active stations to GeoPackage\n")
print(f"   (Time series data will be fetched later as needed)\n")

# Step 1: Copy scope regions
print(f"Step 1: Copying scope regions...")
scope_regions.to_file(OUTPUT_GPKG, layer=SCOPE_LAYER, driver='GPKG')
print(f"   ✅ Saved layer: {SCOPE_LAYER} ({len(scope_regions):,} regions)")

# Step 2: Save active stations for each buffer distance
print(f"\nStep 2: Saving active water station layers...\n")

for buffer_m in BUFFER_DISTANCES:
    joined = buffer_results[buffer_m]['joined']
    
    # Filter to ACTIVE stations only (those with latest measurement >= 2015)
    active_joined = joined[joined['CODE'].isin(active_stations)].copy()
    
    # Add a flag indicating these are active
    active_joined['is_active_2015plus'] = True
    
    # Add a note about data availability
    active_joined['data_note'] = 'Historical data available via API (query year-by-year)'
    
    # Drop FID column if it exists (GeoPackage creates its own)
    if 'FID' in active_joined.columns:
        active_joined = active_joined.drop(columns=['FID'])
    
    layer_name = f"water_stations_{buffer_m}m"
    
    # Save to GeoPackage
    active_joined.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG')
    
    unique_stations = active_joined['CODE'].nunique()
    regions_covered = active_joined[id_col].nunique()
    
    print(f"   ✅ {layer_name}")
    print(f"      • {unique_stations} active stations")
    print(f"      • {regions_covered:,} regions covered")
    print(f"      • Columns: CODE, NAAM, TIJDSTIP_LAATSTE_METING, PARAMETER_WAT_OMSCHRIJVING, geometry, etc.")

print(f"\n" + "="*80)
print(f"✅ GEOPACKAGE SAVED SUCCESSFULLY!")
print(f"="*80)
print(f"\n📂 Location: {OUTPUT_GPKG}")
print(f"\n📊 Summary:")
print(f"   • {len(scope_regions):,} scope regions")
print(f"   • 64 active water level stations (2015+)")
print(f"   • 4 buffer layers: 0m, 50m, 100m, 200m")

print(f"\n🎯 What's included for each station:")
print(f"   • CODE: Station identifier (for API queries)")
print(f"   • NAAM: Station name")
print(f"   • TIJDSTIP_LAATSTE_METING: Latest measurement timestamp")
print(f"   • PARAMETER_WAT_OMSCHRIJVING: Parameter description")
print(f"   • geometry: Point location (EPSG:28992)")
print(f"   • is_active_2015plus: Flag (all True)")

print(f"\n💡 Next steps:")
print(f"   1. Open in QGIS and visualize coverage")
print(f"   2. Choose buffer distance (100m recommended)")
print(f"   3. In DataHandler:")
print(f"      - Load water_stations_100m layer")
print(f"      - For each scope region, find nearby stations")
print(f"      - Use fetch_historical_data_chunked() to get time series")
print(f"      - Extract high water event features")

print(f"\n⚡ Time saved: ~8 minutes (skipped full time series query)")
print(f"   You can query specific stations as needed later!")

📦 QUICK EXPORT: Saving 64 active stations to GeoPackage

   (Time series data will be fetched later as needed)

Step 1: Copying scope regions...
   ✅ Saved layer: vlakken_scope (12,130 regions)

Step 2: Saving active water station layers...

   ✅ water_stations_0m
      • 64 active stations
      • 63 regions covered
      • Columns: CODE, NAAM, TIJDSTIP_LAATSTE_METING, PARAMETER_WAT_OMSCHRIJVING, geometry, etc.
   ✅ water_stations_50m
      • 64 active stations
      • 213 regions covered
      • Columns: CODE, NAAM, TIJDSTIP_LAATSTE_METING, PARAMETER_WAT_OMSCHRIJVING, geometry, etc.
   ✅ water_stations_100m
      • 64 active stations
      • 340 regions covered
      • Columns: CODE, NAAM, TIJDSTIP_LAATSTE_METING, PARAMETER_WAT_OMSCHRIJVING, geometry, etc.
   ✅ water_stations_200m
      • 64 active stations
      • 608 regions covered
      • Columns: CODE, NAAM, TIJDSTIP_LAATSTE_METING, PARAMETER_WAT_OMSCHRIJVING, geometry, etc.

✅ GEOPACKAGE SAVED SUCCESSFULLY!

📂 Location: /Users/

## Reference: How to Query Time Series Later

Use this code snippet when you need to fetch historical data for specific stations.

In [ ]:
"""
REFERENCE CODE - Use this later when you need time series data:

# Example: Fetch historical data for a single station
station_code = "steyl"
df = fetch_historical_data_chunked(station_code, start_year=2016, end_year=2025)

if df is not None:
    print(f"Retrieved {len(df):,} measurements")
    print(f"Years: {df['year'].min()}-{df['year'].max()}")
    
    # Extract high water events (example: 95th percentile)
    threshold = df['water_level_cm'].quantile(0.95)
    high_water_events = df[df['water_level_cm'] > threshold]
    print(f"High water events (>95th percentile): {len(high_water_events)}")

# Example: Query multiple stations for a region
station_codes = ["steyl", "venlo", "well"]
all_data = []
for code in station_codes:
    df = fetch_historical_data_chunked(code, start_year=2020, end_year=2024)
    if df is not None:
        all_data.append(df)

combined = pd.concat(all_data) if all_data else None

# Remember: 
# - Each station takes ~2-3 seconds to query (10 years)
# - Query only the stations you need for each region
# - Cache results to avoid re-querying
"""

print("📖 Reference code added above (commented out)")
print("   Copy and adapt when you implement DataHandler features")

In [ ]:
# Quick reference: List of all 64 active stations
print(f"📋 Complete list of 64 active stations (2015+):\n")
print(f"   (All are within 200m of your scope regions)\n")

for i, station in enumerate(sorted(active_stations), 1):
    print(f"   {i:2d}. {station}")

print(f"\n💾 These stations are saved in the GeoPackage layers:")
print(f"   • water_stations_0m")
print(f"   • water_stations_50m")
print(f"   • water_stations_100m")
print(f"   • water_stations_200m")